# Vehicle routing

Route a fleet over `ortidy`'s bundled locations. We build a Euclidean distance matrix with `ortidy.distance_matrix`, reused across both examples. Each solve returns an **edge-flow** frame of trips.

In [ ]:
import plotly.express as px

import ortidy
from ortidy import distance_matrix, solve_routing

locations = ortidy.data.locations()
locations["locationId"] = locations.index

matrix = distance_matrix(
    locations, method="euclidean", id_column="locationId"
)
# Node labels in the matrix are strings; mirror that for plotting joins.
coords = locations.assign(nodeId=locations["locationId"].astype(str))

## 1. Multiple vehicles from one depot

See https://developers.google.com/optimization/routing/vrp

In [ ]:
result = solve_routing(matrix, vehicles=4)
print(result.status, "| total distance:", result.objective)

plot = result.frame.merge(coords, left_on="departure", right_on="nodeId")
fig = px.line(
    plot,
    x="x",
    y="y",
    color="vehicleId",
    hover_data=["departure", "destination", "distance"],
    markers=True,
    height=600,
    title="4 vehicles from one depot",
)
fig

## 2. Capacitated fleet with per-location demand

Each vehicle has a capacity and each location a demand; the result frame gains a `load` column. See https://developers.google.com/optimization/routing/cvrp

In [ ]:
vehicles = ortidy.data.vehicles(with_capacity=True)
vehicles["vehicleId"] = vehicles.index

result = solve_routing(
    matrix, vehicles=vehicles, locations=locations, starting_point=0
)
print(result.status, "| total distance:", result.objective)

plot = result.frame.merge(coords, left_on="departure", right_on="nodeId")
fig = px.line(
    plot,
    x="x",
    y="y",
    color="vehicleId",
    hover_data=["departure", "destination", "distance", "load"],
    markers=True,
    height=600,
    title="Capacitated VRP",
)
fig

## Optional visits (prize-collecting)

Make stops droppable at a penalty — the solver skips ones too costly to serve. Dropped nodes are reported in `metadata['dropped']`.

In [ ]:
# penalties accepts a {node: penalty} mapping (no pandas needed here)
penalties = {node: 300 for node in range(1, len(locations))}
result = ortidy.solve_routing(matrix, vehicles=1, penalties=penalties)
print(result.status, "| objective:", result.objective)
print("dropped:", result.metadata["dropped"])